# uv Dependency Resolution Lab

This notebook teaches how `uv` resolves, locks, and synchronizes dependencies by using one consistent example project: **weather-dashboard**. The focus is dependency management and reproducibility, not application programming.

## Learning objectives

By the end of this notebook, you will be able to:

- Explain how `uv` creates and manages a project environment.
- Describe the relationship between `pyproject.toml`, `.venv`, and `uv.lock`.
- Distinguish dependency resolution, dependency locking, and environment synchronization.
- Update dependencies safely with `uv lock --upgrade` and `uv lock --upgrade-package`.
- Use `uv sync --frozen` and `uv lock --check` in CI/CD pipelines.
- Apply dependency groups, resolution strategies, and platform-specific markers.

> **Note**: This notebook creates and resets the local `weather-dashboard` example project so the cells can be rerun from top to bottom.

## Mental model

Modern Python dependency management has three separate jobs. Resolution chooses a valid dependency graph, locking records that graph, and synchronization installs the locked graph into an environment. Keeping these steps separate makes local development and CI/CD workflows more predictable.

```mermaid
flowchart LR
    A[pyproject.toml\nDeclared requirements] --> B[uv resolution\nChoose compatible versions]
    B --> C[uv.lock\nResolved dependency graph]
    C --> D[uv sync\nInstall into .venv]
    D --> E[Reproducible runtime\nLocal, CI, production]
```

> **Tip**: Think of `pyproject.toml` as the contract, `uv.lock` as the resolved plan, and `.venv` as the installed result.

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import textwrap


def find_lab_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / 'projects' / 'proj7_jnotebook',
        cwd.parent if cwd.name == 'weather-dashboard' else cwd,
    ]
    for candidate in candidates:
        if candidate.name == 'proj7_jnotebook' or (candidate / 'weather-dashboard').exists():
            return candidate
    return cwd / 'projects' / 'proj7_jnotebook'


LAB_ROOT = find_lab_root()
WEATHER_PROJECT = LAB_ROOT / 'weather-dashboard'

BASE_PYPROJECT = '''[project]
name = "weather-dashboard"
version = "0.1.0"
requires-python = ">=3.11"

dependencies = [
    "fastapi>=0.115",
    "httpx>=0.28",
    "pydantic>=2.10",
    "rich>=14.0",
]

[dependency-groups]
dev = [
    "pytest",
    "ruff",
    "mypy",
]
'''

PROJECT_README = '''# Weather Dashboard

`weather-dashboard` is a fictional project used by the uv dependency resolution notebook.
It keeps Python code intentionally small so the lab can focus on dependency behavior.
'''

INIT_FILE = '''__all__ = ["__version__"]

__version__ = "0.1.0"
'''

SETTINGS_FILE = '''DEFAULT_CITY = "Berlin"
DEFAULT_UNITS = "metric"
'''

TEST_FILE = '''from weather_dashboard.settings import DEFAULT_CITY, DEFAULT_UNITS


def test_default_settings():
    assert DEFAULT_CITY == "Berlin"
    assert DEFAULT_UNITS == "metric"
'''


def run_cmd(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    cwd = cwd or WEATHER_PROJECT
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


def run_uv(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    return run_cmd(['uv', *args], cwd=cwd, check=check)


def write_pyproject(content: str = BASE_PYPROJECT) -> None:
    (WEATHER_PROJECT / 'pyproject.toml').write_text(content, encoding='utf-8')


def create_project_scaffold() -> None:
    (WEATHER_PROJECT / 'src' / 'weather_dashboard').mkdir(parents=True, exist_ok=True)
    (WEATHER_PROJECT / 'tests').mkdir(parents=True, exist_ok=True)
    (WEATHER_PROJECT / 'README.md').write_text(PROJECT_README, encoding='utf-8')
    (WEATHER_PROJECT / 'src' / 'weather_dashboard' / '__init__.py').write_text(INIT_FILE, encoding='utf-8')
    (WEATHER_PROJECT / 'src' / 'weather_dashboard' / 'settings.py').write_text(SETTINGS_FILE, encoding='utf-8')
    (WEATHER_PROJECT / 'tests' / 'test_settings.py').write_text(TEST_FILE, encoding='utf-8')


def read_pyproject() -> str:
    return (WEATHER_PROJECT / 'pyproject.toml').read_text(encoding='utf-8')


def show_file(relative_path: str, max_lines: int = 80) -> None:
    path = WEATHER_PROJECT / relative_path
    print(f"--- {relative_path} ---")
    lines = path.read_text(encoding='utf-8').splitlines()
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more lines)")


print(f"Lab root: {LAB_ROOT}")
print(f"Example project: {WEATHER_PROJECT}")

## 1. Project setup

A project needs a declared dependency contract before a resolver can do useful work. `uv init` creates the project shell, `.python-version` pins the interpreter choice, and `uv sync` creates the `.venv` environment from the declared dependencies.

This lab writes the same `pyproject.toml` each time so the demonstration starts from a known baseline.

> **Warning**: The next cell removes and recreates only the local `weather-dashboard` sandbox inside this lab folder.

In [ ]:
LAB_ROOT.mkdir(parents=True, exist_ok=True)
if WEATHER_PROJECT.exists():
    shutil.rmtree(WEATHER_PROJECT)

run_uv(['init', 'weather-dashboard', '--bare', '--name', 'weather-dashboard'], cwd=LAB_ROOT)
write_pyproject()
create_project_scaffold()
(WEATHER_PROJECT / '.python-version').write_text('3.11\n', encoding='utf-8')

run_uv(['python', 'install', '3.11'], cwd=WEATHER_PROJECT)
run_uv(['sync'], cwd=WEATHER_PROJECT)

show_file('pyproject.toml')
show_file('.python-version')

Expected output includes messages similar to:

```text
$ uv init weather-dashboard --bare --name weather-dashboard
Initialized project `weather-dashboard`
$ uv python install 3.11
Installed Python 3.11.x
$ uv sync
Resolved ... packages
Prepared ... packages
Installed ... packages
```

After this step:

- `pyproject.toml` contains the direct dependency requirements.
- `.python-version` tells `uv` which interpreter family to use.
- `.venv/` contains the synchronized project environment.
- `uv.lock` records the resolved dependency graph.

## 2. Understanding `uv.lock`

A lockfile records the exact versions and artifact metadata selected by the resolver. This prevents every developer, CI runner, or deployment machine from resolving a slightly different environment.

> **Note**: Commit `uv.lock` to version control for applications and training projects. It makes dependency changes reviewable and reproducible.

Traditional `pip freeze` captures what happened to be installed locally. A lockfile records a resolved graph produced from project requirements.

In [ ]:
run_uv(['lock', '--check'])
show_file('uv.lock', max_lines=45)

How the files relate:

- `pyproject.toml` stays human-authored and concise.
- `uv.lock` becomes detailed because it stores direct and transitive dependencies.
- `.venv` is disposable because `uv sync` can recreate it from `uv.lock`.

```mermaid
flowchart TB
    P[pyproject.toml\nDirect constraints] --> L[uv.lock\nExact resolved graph]
    L --> V[.venv\nInstalled packages]
    V -. can be deleted and recreated .-> L
```

## 3. Updating dependencies through `pyproject.toml`

Dependency changes should start in `pyproject.toml` because that file describes what the project needs. After editing it, the lockfile must be refreshed so the resolved graph matches the new requirements.

In this example, the weather dashboard adds `orjson` as a realistic high-performance JSON dependency.

In [ ]:
pyproject = read_pyproject()
if '"orjson>=3.10"' not in pyproject:
    pyproject = pyproject.replace('    "rich>=14.0",\n]', '    "rich>=14.0",\n    "orjson>=3.10",\n]')
    write_pyproject(pyproject)

show_file('pyproject.toml')
run_uv(['lock', '--check'], check=False)
run_uv(['lock'])

Expected behavior:

```text
$ uv lock --check
The lockfile needs to be updated...
$ uv lock
Resolved ... packages
```

Best practice: review `pyproject.toml` and `uv.lock` together in code review. One explains the requested change; the other shows the resolved result.

## 4. `uv lock --upgrade`

Use `uv lock --upgrade` when you intentionally want to refresh all dependencies to the newest versions allowed by `pyproject.toml`. This is useful for scheduled maintenance, but it can change many packages at once.

> **Tip**: Treat broad upgrades as planned work. Run tests after the lockfile changes.

In [ ]:
run_uv(['lock', '--upgrade'])

## 5. `uv lock --upgrade-package`

Use `uv lock --upgrade-package` when you want a smaller, targeted update. It is a better fit for security fixes or controlled dependency bumps because the rest of the lockfile changes less.

In [ ]:
run_uv(['lock', '--upgrade-package', 'httpx'])

## 6. Synchronizing environments with `uv sync`

Locking decides what should be installed. Synchronization makes the `.venv` match that decision. `uv sync` is the everyday local-development command after cloning a repository or changing dependencies.

In [ ]:
run_uv(['sync'])
show_file('.venv/pyvenv.cfg', max_lines=20)

After `uv sync`:

- `.venv` contains installed packages matching `uv.lock`.
- `pyvenv.cfg` records the interpreter used by the environment.
- If dependency constraints changed, `uv` may update `uv.lock` unless you use frozen or locked modes.

> **Common pitfall**: Do not commit `.venv`. Commit `pyproject.toml` and `uv.lock`, then recreate `.venv` with `uv sync`.

## 7. Reproducible installations with `uv sync --frozen`

`uv sync --frozen` installs from the existing lockfile without updating it. This is preferred in CI/CD because the build should use the reviewed lockfile, not silently produce a new dependency graph.

In [ ]:
run_uv(['sync', '--frozen'])

## 8. Lockfile validation with `uv lock --check`

`uv lock --check` verifies that `uv.lock` is consistent with `pyproject.toml`. It is fast and useful as an early CI gate before tests run.

> **CI recommendation**: Use `uv lock --check` to fail quickly when a dependency declaration changed but the lockfile was not committed.

In [ ]:
run_uv(['lock', '--check'])

## 9. Resolution strategies

By default, `uv` prefers the newest compatible versions. Compatibility testing sometimes needs the opposite: the lowest versions allowed by your constraints.

Use `--resolution lowest` to test the full lower bound of the graph. Use `--resolution lowest-direct` to test only direct dependency lower bounds while keeping transitive dependencies newer.

In [ ]:
run_uv(['sync', '--resolution', 'lowest'])
run_uv(['sync', '--resolution', 'lowest-direct'])

# Restore the normal latest-compatible lock after the demonstration.
run_uv(['lock', '--upgrade'])
run_uv(['sync'])

Best practices for resolution strategies:

- Use default resolution for normal development.
- Add a separate CI compatibility lane for `lowest` or `lowest-direct`.
- Use lower-bound testing mainly for libraries or shared internal packages.
- Restore the normal lockfile policy after compatibility experiments.

## 10. Dependency groups

Dependency groups separate tools needed for development from dependencies needed by the application. The weather dashboard uses a `dev` group for `pytest`, `ruff`, and `mypy`.

> **Tip**: Keep production dependencies small, and place test, lint, type-checking, and documentation tools in groups.

In [ ]:
run_uv(['sync', '--group', 'dev'])
run_uv(['run', 'ruff', '--version'])
run_uv(['run', 'mypy', '--version'])

## 11. Platform-specific dependency markers

Some packages only exist for certain operating systems or Python versions. Dependency markers express those constraints directly in `pyproject.toml`, allowing one project to resolve correctly on different platforms.

This example adds `pywin32` only for Windows. Linux and macOS environments keep a valid graph because the marker excludes the package there.

In [ ]:
pyproject = read_pyproject()
marker_dependency = '    "pywin32>=306; sys_platform == \"win32\"",'
if 'pywin32>=306' not in pyproject:
    pyproject = pyproject.replace('    "orjson>=3.10",\n]', f'    "orjson>=3.10",\n{marker_dependency}\n]')
    write_pyproject(pyproject)

show_file('pyproject.toml')
run_uv(['lock'])
run_uv(['sync'])

Marker guidance:

- Use PEP 508 markers such as `sys_platform == "win32"` or `python_version >= "3.12"`.
- Prefer markers over conditional install instructions in README files.
- Test the lockfile on the platforms you claim to support.
- Keep platform-specific dependencies as narrow as possible.

## 12. Recommended commands for development and CI/CD

Use different commands depending on whether you are changing dependencies, installing an already-reviewed environment, or validating repository state.

| Scenario | Recommended command | Why |
|---|---|---|
| Start local development after cloning | `uv sync` | Creates `.venv` from the lockfile and project metadata. |
| Add or edit dependencies | Edit `pyproject.toml`, then `uv lock` | Keeps declared requirements and resolved graph aligned. |
| Scheduled dependency refresh | `uv lock --upgrade` | Updates all packages within allowed constraints. |
| Targeted security or bug-fix update | `uv lock --upgrade-package <name>` | Limits lockfile churn to the selected dependency and its needed graph changes. |
| CI install step | `uv sync --frozen` | Installs without rewriting `uv.lock`. |
| CI validation step | `uv lock --check` | Fails if `pyproject.toml` and `uv.lock` disagree. |
| Compatibility testing | `uv sync --resolution lowest` or `lowest-direct` | Verifies lower-bound dependency claims. |
| Install development tools | `uv sync --group dev` | Installs test, lint, and type-check tools when needed. |

In [ ]:
run_uv(['lock', '--check'])
run_uv(['sync', '--frozen'])
print('The weather-dashboard dependency state is locked, synchronized, and CI-ready.')

## Key takeaways

- `pyproject.toml` defines dependency intent.
- `uv.lock` records the exact resolved dependency graph and should be committed.
- `.venv` is a generated environment and should not be committed.
- `uv sync` is for local synchronization; `uv sync --frozen` is safer for CI.
- `uv lock --check` is a fast guardrail that keeps dependency changes honest.
- Resolution strategies, dependency groups, and markers help one project support multiple development and deployment realities.